In [ ]:
import argparse
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, count, round, col, regexp_replace

In [ ]:
spark = SparkSession.builder.appName("review-per-listing").getOrCreate()
spark

In [ ]:
listing_data = "../data-src/listings.csv.gz"
reviews_data = "../data-src/reviews.csv.gz"

spark = SparkSession.builder.appName("review-per-listing").getOrCreate()

listings = spark.read.csv(
    listing_data,
    header=True,
    inferSchema=True,
    multiLine=True,
    sep=",",
    quote='"',
    escape='"',
    mode="PERMISSIVE",
)

reviews = spark.read.csv(
    reviews_data,
    header=True,
    inferSchema=True,
    multiLine=True,
    sep=",",
    escape='"',
    quote='"',
    mode="PERMISSIVE"
)

In [ ]:
listings.count(), reviews.count()

In [ ]:
listings = listings.select("id", "name", "price", "source")
listings = listings.withColumnRenamed("id", "listing_id")
listings.show(5)

In [ ]:

reviews.show(5)

In [ ]:
reviews_per_listing = listings.join(reviews, on="listing_id", how="left")
reviews_per_listing.show(5)

In [ ]:
reviews_per_listing = reviews_per_listing.withColumn(
    "price", regexp_replace("price", "[$,]", "").cast("double")
)
reviews_per_listing_export = reviews_per_listing.groupBy(
    ["listing_id", "name", "source"]
).agg(
    round(avg("price"), 2).alias("price"),
    count("*").alias("mum_reviews"),
)

# reviews_per_listing_export = reviews_per_listing_export.limit(30000)

reviews_per_listing_export.show(5)

In [ ]:
reviews_per_listing_export.write.format("csv").mode("overwrite").save("./output")